# 3장 웹부록 | 손글씨 숫자를 읽는 신경망

「영화로 보는 AI 리터러시」 3장 '기계는 어떻게 배우는가: 머신러닝과 딥러닝'의 코랩 실습입니다.

손으로 쓴 숫자 사진(MNIST)으로 작은 신경망과 CNN을 차례로 학습시킵니다. 학습이 진행될수록 손실 그래프가 내려가는 모습을 확인하고, 모델이 틀린 숫자들을 모아 봅니다. 마지막에는 내가 그린 숫자를 넣어 맞히는지 시험합니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 곳만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 전체를 실행하는 데 1~2분쯤 걸립니다(대부분 4단계 CNN 학습 시간). 그래프 글자는 한글이 깨지지 않도록 영어로 적었습니다.

## 1단계. 손글씨 숫자 사진 불러오기

MNIST는 손으로 쓴 0~9 숫자 사진 7만 장을 모은 유명한 데이터입니다. 사진 한 장은 가로세로 28칸, 모두 784개의 점(픽셀)입니다.
코랩에는 이 가운데 **3만 장(학습용 2만 장, 시험용 1만 장)**이 `sample_data` 폴더에 미리 들어 있어서, 인터넷에서 따로 내려받을 필요가 없습니다.

공부에 쓰는 사진(학습용)과 시험에 쓰는 사진(시험용)을 반드시 나눕니다(3-7절). 시험용 사진은 모델이 학습 중에 한 번도 보지 못합니다.

(코랩이 아닌 곳에서 열면 이 파일이 없습니다. 그럴 때는 scikit-learn에 들어 있는 8×8 크기의 작은 손글씨 숫자 1,797장으로 대신합니다.)

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt

TRAIN_CSV = "/content/sample_data/mnist_train_small.csv"
TEST_CSV = "/content/sample_data/mnist_test.csv"

if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
    train = np.loadtxt(TRAIN_CSV, delimiter=",", dtype=np.float32)
    test = np.loadtxt(TEST_CSV, delimiter=",", dtype=np.float32)
    X_train, y_train = train[:, 1:] / 255.0, train[:, 0].astype(int)
    X_test, y_test = test[:, 1:] / 255.0, test[:, 0].astype(int)
    SIDE = 28
    DATA_NAME = "MNIST (Colab sample_data)"
else:
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    d = load_digits()
    X = (d.data / 16.0).astype(np.float32)
    X_train, X_test, y_train, y_test = train_test_split(X, d.target, test_size=0.3, random_state=0)
    SIDE = 8
    DATA_NAME = "sklearn digits 8x8"

print(f"데이터: {DATA_NAME}")
print(f"학습용 {len(X_train):,}장, 시험용 {len(X_test):,}장, 사진 크기 {SIDE}×{SIDE}")

fig, axes = plt.subplots(2, 10, figsize=(10, 2.4))
for ax, img, label in zip(axes.ravel(), X_train[:20], y_train[:20]):
    ax.imshow(img.reshape(SIDE, SIDE), cmap="gray_r")
    ax.set_title(str(label))
    ax.axis("off")
plt.suptitle("Training images and their labels (answers)")
plt.tight_layout()
plt.show()

컴퓨터에게 사진은 숫자의 표일 뿐입니다. 아래는 사진 한 장을 숫자로 본 모습입니다(0은 흰 바탕, 1에 가까울수록 진한 잉크). 1장에서 본 것처럼, 모든 것은 숫자가 됩니다.

In [ ]:
np.set_printoptions(linewidth=200)
sample = X_train[0].reshape(SIDE, SIDE)
step = 2 if SIDE == 28 else 1           # 28칸은 너무 넓어서 한 칸씩 건너뛰어 보여 줍니다
print(f"정답: {y_train[0]}")
print((sample[::step, ::step] * 9).round().astype(int))

## 2단계. 작은 신경망 학습시키기

입력층(784칸) → 은닉층(뉴런 64개) → 출력층(숫자 10개)으로 된 작은 신경망입니다(그림 3-7).
학습용 사진을 한 번 다 보는 것을 '한 바퀴(에포크)'라고 합니다. 한 바퀴를 돌 때마다 **손실**(얼마나 틀렸는지)과 시험용 사진의 정답률을 기록합니다.

손실을 줄이는 쪽으로 가중치를 조금씩 고치는 것이 경사 하강법이고, 그 고칠 양을 층마다 거꾸로 계산하는 것이 역전파입니다(3-3절, 3-5절).

👉 **바꿔 볼 곳**: `HIDDEN`(은닉층 뉴런 수), `EPOCHS`(바퀴 수), `LEARNING_RATE`(학습률, 보폭)를 바꿔 보세요. 학습률을 `0.5`처럼 크게 하면 어떻게 될까요?

In [ ]:
import warnings
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

HIDDEN = 64            # ← 은닉층 뉴런 수
EPOCHS = 10            # ← 몇 바퀴 학습할지
LEARNING_RATE = 0.001  # ← 학습률(보폭). 0.01, 0.5 등으로 바꿔 보세요

mlp = MLPClassifier(hidden_layer_sizes=(HIDDEN,), learning_rate_init=LEARNING_RATE,
                    batch_size=64, random_state=0)
mlp_loss, mlp_acc = [], []
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    mlp.partial_fit(X_train, y_train, classes=np.arange(10))   # 학습용 사진을 한 바퀴 봅니다
    acc = mlp.score(X_test, y_test)                            # 시험용 사진으로 정답률을 잽니다
    mlp_loss.append(mlp.loss_)
    mlp_acc.append(acc)
    print(f"{epoch:2d}바퀴: 손실 {mlp.loss_:.3f}   시험 정답률 {acc:.1%}")
print(f"걸린 시간 {time.time() - t0:.0f}초")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(range(1, EPOCHS + 1), mlp_loss, "o-")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss goes down as it learns")
ax2.plot(range(1, EPOCHS + 1), mlp_acc, "o-", color="#2ca02c")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Test accuracy")
ax2.set_title("Accuracy on unseen test images")
plt.tight_layout()
plt.show()

## 3단계. 모델이 틀린 숫자 모아 보기

시험용 사진 가운데 신경망이 틀린 것들을 모읍니다. 제목의 `pred`는 모델의 답, `true`는 정답입니다.
사람이 봐도 헷갈리는 숫자가 많습니까, 아니면 사람에게는 쉬운데 모델만 틀린 숫자가 많습니까?

In [ ]:
def show_mistakes(pred, title, n=18):
    wrong = np.where(pred != y_test)[0]
    print(f"시험용 {len(y_test):,}장 가운데 {len(wrong)}장을 틀렸습니다 (정답률 {1 - len(wrong) / len(y_test):.1%}).")
    cols = 9
    rows = int(np.ceil(min(n, len(wrong)) / cols))
    if rows == 0:
        return
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.1, rows * 1.4))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, i in zip(np.ravel(axes), wrong[:n]):
        ax.imshow(X_test[i].reshape(SIDE, SIDE), cmap="gray_r")
        ax.set_title(f"pred {pred[i]}\ntrue {y_test[i]}", fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

mlp_pred = mlp.predict(X_test)
show_mistakes(mlp_pred, "Mistakes of the small neural network")

## 4단계. CNN 학습시키기

이번에는 **합성곱 신경망(CNN)**입니다(3-6절). 3×3 크기의 작은 **필터** 8개가 사진을 한 칸씩 훑으며 무늬를 찾습니다.
책의 심화 상식에서 본 세 단계를 그대로 거칩니다.

1. **합성곱**: 필터를 옮기며 곱해 더해 '특징 지도'를 만듭니다.
2. **활성화**: 음수는 0으로 바꿉니다.
3. **풀링**: 2×2 칸마다 가장 큰 값만 남겨 지도를 절반으로 줄입니다.

그다음 출력층이 숫자 10개 중 하나를 고릅니다. 아래 칸은 이 CNN을 numpy로 직접 만든 것입니다. 실행만 하면 됩니다(아무것도 나오지 않는 것이 정상입니다).

In [ ]:
class TinyCNN:
    """필터 8개짜리 합성곱 층 1개 + 풀링 + 출력층으로 된 작은 CNN (numpy로 직접 구현)"""

    def __init__(self, side, n_filters=8, seed=0):
        rng = np.random.default_rng(seed)
        self.side, self.F = side, n_filters
        self.Wc = rng.normal(0, np.sqrt(2 / 9), (9, n_filters)).astype(np.float32)   # 3×3 필터들
        self.bc = np.zeros(n_filters, np.float32)
        self.ho = (side - 2) // 2                                                  # 풀링 뒤 크기
        D = self.ho * self.ho * n_filters
        self.Wd = rng.normal(0, np.sqrt(1 / D), (D, 10)).astype(np.float32)        # 출력층
        self.bd = np.zeros(10, np.float32)

    def _patches(self, X):
        imgs = X.reshape(-1, self.side, self.side)
        p = np.lib.stride_tricks.sliding_window_view(imgs, (3, 3), axis=(1, 2))
        n = 2 * self.ho                                    # 풀링이 딱 나눠지도록 맞춥니다
        return p[:, :n, :n].reshape(len(X), n * n, 9)

    def forward(self, X):
        P = self._patches(X)
        n = 2 * self.ho
        conv = (P @ self.Wc + self.bc).reshape(len(X), n, n, self.F)       # 1) 합성곱
        act = np.maximum(conv, 0)                                          # 2) 활성화
        blocks = act.reshape(len(X), self.ho, 2, self.ho, 2, self.F)
        pooled = blocks.max(axis=(2, 4))                                   # 3) 풀링
        logits = pooled.reshape(len(X), -1) @ self.Wd + self.bd            # 출력층
        return logits, (P, conv, blocks, pooled)

    def predict(self, X, batch=1000):
        return np.concatenate([self.forward(X[i:i + batch])[0].argmax(1) for i in range(0, len(X), batch)])

    def train_step(self, X, y, lr):
        logits, (P, conv, blocks, pooled) = self.forward(X)
        z = logits - logits.max(1, keepdims=True)
        prob = np.exp(z) / np.exp(z).sum(1, keepdims=True)
        B = len(X)
        loss = -np.log(prob[np.arange(B), y] + 1e-9).mean()                # 손실
        # ---- 역전파: 틀린 책임을 뒤에서부터 나눕니다 ----
        g = prob.copy()
        g[np.arange(B), y] -= 1
        g /= B
        flat = pooled.reshape(B, -1)
        gWd, gbd = flat.T @ g, g.sum(0)
        gpool = (g @ self.Wd.T).reshape(pooled.shape)
        mask = blocks == pooled[:, :, None, :, None, :]
        gact = (mask * gpool[:, :, None, :, None, :]).reshape(conv.shape)
        gconv = (gact * (conv > 0)).reshape(B, -1, self.F)
        gWc = P.reshape(-1, 9).T @ gconv.reshape(-1, self.F)
        gbc = gconv.sum((0, 1))
        # ---- 경사 하강법: 손실이 줄어드는 쪽으로 조금씩 ----
        self.Wd -= lr * gWd; self.bd -= lr * gbd
        self.Wc -= lr * gWc; self.bc -= lr * gbc
        return loss

print("CNN 준비 완료!")

이제 학습시킵니다. 사진을 64장씩 묶어 보여 주고, 묶음마다 손실을 기록합니다.

👉 **바꿔 볼 곳**: `CNN_EPOCHS`(바퀴 수), `CNN_LR`(학습률)

In [ ]:
CNN_EPOCHS = 3 if SIDE == 28 else 30   # ← 몇 바퀴 학습할지 (작은 8×8 데이터는 사진이 적어 더 많이 돕니다)
CNN_LR = 0.1        # ← 학습률(보폭)
BATCH = 64

cnn = TinyCNN(SIDE)
rng = np.random.default_rng(0)
cnn_loss = []
t0 = time.time()
for epoch in range(1, CNN_EPOCHS + 1):
    order = rng.permutation(len(X_train))
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        cnn_loss.append(cnn.train_step(X_train[idx], y_train[idx], CNN_LR))
    acc = (cnn.predict(X_test) == y_test).mean()
    print(f"{epoch}바퀴: 최근 손실 {np.mean(cnn_loss[-50:]):.3f}   시험 정답률 {acc:.1%}")
print(f"걸린 시간 {time.time() - t0:.0f}초")

smooth = np.convolve(cnn_loss, np.ones(20) / 20, mode="valid")
plt.figure(figsize=(8, 3.5))
plt.plot(cnn_loss, alpha=0.3, label="each batch")
plt.plot(np.arange(len(smooth)) + 10, smooth, label="moving average")
plt.xlabel("Training step (batch of 64 images)")
plt.ylabel("Loss")
plt.title("CNN loss goes down step by step")
plt.legend()
plt.show()

### CNN이 배운 필터와 특징 지도

처음에는 무작위였던 3×3 필터들이 학습을 거치며 어떤 무늬(세로선, 가로선, 비스듬한 선)에 반응하도록 바뀌었습니다.
아래쪽은 사진 한 장을 필터 8개로 훑어 만든 **특징 지도**입니다. 밝은 곳이 '여기에 내 무늬가 있다'고 반응한 자리입니다.

In [ ]:
img = X_test[:1]
_, (P, conv, blocks, pooled) = cnn.forward(img)
fig, axes = plt.subplots(2, 9, figsize=(12, 3.2))
axes[0, 0].imshow(img.reshape(SIDE, SIDE), cmap="gray_r"); axes[0, 0].set_title("input")
axes[1, 0].axis("off")
for k in range(8):
    axes[0, k + 1].imshow(cnn.Wc[:, k].reshape(3, 3), cmap="RdBu_r")
    axes[0, k + 1].set_title(f"filter {k + 1}", fontsize=8)
    axes[1, k + 1].imshow(np.maximum(conv[0, :, :, k], 0), cmap="magma")
    axes[1, k + 1].set_title(f"feature map {k + 1}", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

### 두 모델 비교하기

이 CNN은 필터 8개, 합성곱 층 1개뿐인 아주 작은 모델이라 늘 이기지는 않습니다. 실제로 쓰이는 CNN은 이 세 단계를 여러 번 쌓고 필터도 수백 개를 씁니다. 그래도 필터 몇 개만으로 숫자를 꽤 잘 읽는다는 점이 놀랍습니다.

In [ ]:
cnn_pred = cnn.predict(X_test)
print(f"작은 신경망 정답률: {(mlp_pred == y_test).mean():.1%}")
print(f"CNN 정답률       : {(cnn_pred == y_test).mean():.1%}")
show_mistakes(cnn_pred, "Mistakes of the CNN")

## 5단계. 내가 그린 숫자 넣어 보기

아래 `MY_DRAWING`에 숫자를 그려 보세요. `#`은 잉크, `.`은 빈칸입니다. 가로 14칸, 세로 14칸입니다.
처음에는 숫자 7이 그려져 있습니다. `#`을 옮겨 다른 숫자를 그리고 실행해 보세요.

컴퓨터는 그림을 MNIST 사진처럼 크기를 맞추고 가운데로 옮긴 뒤 두 모델에게 물어봅니다.

In [ ]:
MY_DRAWING = """
..............
..............
...#########..
..........##..
.........##...
........##....
.......##.....
......##......
......##......
.....##.......
.....##.......
.....##.......
..............
..............
"""   # ← '#'을 옮겨 다른 숫자를 그려 보세요 (14줄, 줄마다 14칸)

def drawing_to_image(text, side):
    rows = [r for r in text.strip().splitlines()]
    grid = np.array([[1.0 if ch == "#" else 0.0 for ch in r.ljust(14)[:14]] for r in rows[:14]])
    ys, xs = np.nonzero(grid)
    if len(ys) == 0:
        return np.zeros(side * side, np.float32)
    crop = grid[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    box = int(round(side * 20 / 28))                   # MNIST처럼 숫자를 20/28 크기 상자에 맞춥니다
    h, w = crop.shape
    scale = box / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    yy = (np.arange(nh) + 0.5) / scale - 0.5           # 간단한 확대·축소 (이중 선형 보간)
    xx = (np.arange(nw) + 0.5) / scale - 0.5
    tmp = np.array([np.interp(xx, np.arange(w), row) for row in crop])
    small = np.array([np.interp(yy, np.arange(h), col) for col in tmp.T]).T
    out = np.zeros((side, side))
    cy, cx = (side - nh) // 2, (side - nw) // 2
    out[cy:cy + nh, cx:cx + nw] = small
    return np.clip(out, 0, 1).ravel().astype(np.float32)

mine = drawing_to_image(MY_DRAWING, SIDE)
probs = mlp.predict_proba(mine[None])[0]
cnn_logits = cnn.forward(mine[None])[0][0]
cnn_probs = np.exp(cnn_logits - cnn_logits.max()); cnn_probs /= cnn_probs.sum()

fig, (a0, a1, a2) = plt.subplots(1, 3, figsize=(11, 3))
a0.imshow(mine.reshape(SIDE, SIDE), cmap="gray_r"); a0.set_title("My drawing"); a0.axis("off")
a1.bar(range(10), probs, color="#5b8def"); a1.set_xticks(range(10)); a1.set_title(f"Neural net says: {probs.argmax()}")
a2.bar(range(10), cnn_probs, color="#e8743b"); a2.set_xticks(range(10)); a2.set_title(f"CNN says: {cnn_probs.argmax()}")
plt.tight_layout()
plt.show()
print(f"작은 신경망의 답: {probs.argmax()} (확신 {probs.max():.0%})")
print(f"CNN의 답       : {cnn_probs.argmax()} (확신 {cnn_probs.max():.0%})")

잘 맞혔나요? 숫자를 한쪽 구석에 작게 그리거나, 아주 굵게 그리면 어떻게 되는지도 시험해 보세요. 학습 사진과 다른 모양일수록 모델은 쉽게 흔들립니다.

## 생각해 볼 것

1. 학습률을 아주 크게(예: 0.5) 했을 때 손실 그래프는 어떻게 되었습니까? 3-3절의 '보폭' 비유로 설명해 보세요.
2. 모델이 틀린 숫자들 가운데 사람도 헷갈릴 만한 것은 몇 개였습니까? 데이터의 '정답'을 붙인 사람도 틀릴 수 있다면, 모델에게는 어떤 영향이 있을까요?
3. 이 모델은 사람에게 '7은 위에 가로선이 있다' 같은 규칙을 배운 적이 없습니다. 그런데도 7을 맞힙니다. 3-1절의 '규칙을 적을까, 예시를 보여 줄까'와 연결해 설명해 보세요.